# Lesson 04: Multi-Layered Perceptrons & Building Your Own Neural Framework

In this notebook, you will:
1. Build a modular NumPy neural network framework (`Linear`, `Tanh`, `Softmax`, `CrossEntropyLoss`, `Net`) from scratch.
2. Train it on a non-linear 2D dataset.
3. Complete the **Lab**: Train your custom framework on handwritten digits (8x8 MNIST).

In [ ]:
import numpy as np
np.random.seed(42)

class Linear:
    def __init__(self, nin, nout):
        self.W = np.random.normal(0, 1.0 / np.sqrt(nin), (nout, nin))
        self.b = np.zeros((1, nout))
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W.T + self.b

    def backward(self, dz):
        self.dW = dz.T @ self.x
        self.db = dz.sum(axis=0, keepdims=True)
        return dz @ self.W

    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db

class Tanh:
    def forward(self, x):
        self.y = np.tanh(x)
        return self.y
    def backward(self, dy):
        return dy * (1.0 - self.y ** 2)

class Softmax:
    def forward(self, z):
        zmax = z.max(axis=1, keepdims=True)
        expz = np.exp(z - zmax)
        self.p = expz / expz.sum(axis=1, keepdims=True)
        return self.p
    def backward(self, dp):
        # Combined with CrossEntropyLoss gradient below
        return dp

class CrossEntropyLoss:
    def forward(self, p, y):
        self.p = p
        self.y = y
        return -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
    def backward(self):
        grad = self.p.copy()
        grad[np.arange(len(self.y)), self.y] -= 1.0
        return grad / len(self.y)

class Net:
    def __init__(self, layers):
        self.layers = layers
    def forward(self, x):
        for l in self.layers:
            x = l.forward(x)
        return x
    def backward(self, grad):
        for l in reversed(self.layers):
            grad = l.backward(grad)
    def update(self, lr):
        for l in self.layers:
            if hasattr(l, 'update'):
                l.update(lr)

## Lab: Training Our Custom Framework on Handwritten Digits

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.data / 16.0
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

net = Net([Linear(64, 32), Tanh(), Linear(32, 10), Softmax()])
loss_fn = CrossEntropyLoss()

for epoch in range(250):
    probs = net.forward(X_tr)
    loss = loss_fn.forward(probs, y_tr)
    net.backward(loss_fn.backward())
    net.update(lr=0.5)
    if (epoch + 1) % 50 == 0:
        acc = (np.argmax(net.forward(X_te), axis=1) == y_te).mean()
        print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f} | Test Acc: {acc*100:.2f}%")